In [1]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# %% [markdown]
# # Enformer Demo: GPU-Accelerated Chromatin Predictions
# 
# Predict chromatin accessibility and histone modifications from DNA sequence
# using DeepMind's Enformer model, visualized in an embedded genome browser.

# %% Setup
!pip install -q kipoiseq pyfaidx igv-notebook tensorflow-hub
!mkdir -p runs

import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'GPU required: Runtime -> Change runtime type -> GPU'
print(f"TensorFlow: {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU')[0].name}")

In [ ]:
# Run model
import tensorflow_hub as hub
import numpy as np
import time
from utils.enformer import (
    download_genome, FastaExtractor, one_hot_encode,
    get_input_interval, get_output_region, export_tracks
)

# Configuration
CHROM = 'chr17'
CENTER = 7_580_000  # TP53 gene

# Prepare sequence
fasta_path = download_genome()
fasta = FastaExtractor(fasta_path)
interval = get_input_interval(CHROM, CENTER)
sequence = one_hot_encode(fasta.extract(interval))

print(f"Target: {CHROM}:{CENTER:,} | Input: {len(sequence):,} bp")

# Load model and run inference
model = hub.load('https://tfhub.dev/deepmind/enformer/1').model

t0 = time.time()
predictions = model.predict_on_batch(sequence[np.newaxis])['human'].numpy()[0]
print(f"Inference: {time.time() - t0:.2f}s | Output: {predictions.shape}")

# Export tracks
track_files = export_tracks(predictions, CHROM, interval.start, './runs/tracks')

In [ ]:
# Visualize
from utils.igv_utils import create_browser, show_interpretation_guide

out_start, out_end = get_output_region(interval.start)
locus = f"{CHROM}:{out_start}-{out_end}"

browser = create_browser(locus, track_files)
show_interpretation_guide()